In [ ]:
run_vamp_analysis = False

if run_vamp_analysis:

    from deeptime.decomposition import VAMP, vamp_score_cv
    aggregated_traj = np.concatenate([distances, dihedrals], axis=1)
    def make_fit_fetch(dim, lagtime):
        def fit_fetch(trajs):
            vamp = VAMP(dim=dim, lagtime=lagtime)
            vamp.fit(trajs)
            return vamp.fetch_model()
        return fit_fetch


    trajs = [[distances], [dihedrals], [aggregated_traj]]  # your raw feature trajectories
    traj_names = ["distances", "dihedrals", "aggregated_traj"]
    results = {}

    for n_vamp in [2,  4, 6,  8,  10]:
        for lagtime in [100, 1000]:
            print(f"Evaluating lagtime = {lagtime}...")
            for i,traj in enumerate(trajs):
                fit_fetch = make_fit_fetch(n_vamp, lagtime)
                scores = vamp_score_cv(fit_fetch, trajs=traj, blocksize=lagtime, n=5, r=2)
                results[(traj_names[i], n_vamp, lagtime)] = scores.mean()
                print(f"features: {traj_names[i]}, dim: {n_vamp}, lagtime={lagtime}: VAMP-2 = {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
if run_vamp_analysis:
    
    import pandas as pd

    rows = []

    for (feature, n_vamp, lag), score in results.items():
        rows.append({
            "feature": feature,
            "n_vamp": n_vamp,
            "lag_frames": lag,
            "score": score,
        })

    df = pd.DataFrame(rows)

    df.to_csv("intermediate_outputs/vamp_feature_selection.csv", index=False)
    df.head()

In [ ]:
if run_vamp_analysis:
    # Bar plot at fixed lag time, dimension
    fig, axes = plt.subplots(1, len([100, 1000]), figsize=(6, 3), sharey=True)

    colors = cm.tab10.colors
    for ax, lag in zip(axes.flatten(), [100, 1000]):

        scores = [results[(feature, 4, lag)] for feature in traj_names]
        ax.bar(traj_names, scores, color=colors[:len(traj_names)])
        ax.set_title(f"lag = {lag}, dim = 4")
        ax.set_ylabel("VAMP-2 score")
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()


    # Plot as a function of dimension, varying lag time, fixed feauture set
    fig, axes = plt.subplots(1, len(traj_names), figsize=(10, 3), sharey=True)
    for ax, feature in zip(axes.flatten(), traj_names):

        scores_100 = [results[(feature, n_vamp, 100)] for n_vamp in [2, 4, 6, 8, 10]]
        scores_1000 = [results[(feature, n_vamp, 1000)] for n_vamp in [2, 4, 6, 8, 10]]

        ax.plot([2, 4, 6, 8, 10], scores_100, marker = "o", label="lag=100")
        ax.plot([2, 4, 6, 8, 10], scores_1000, marker = "s", label="lag=1000")
        ax.set_title(f"features: {feature}")
        ax.set_xlabel("VAMP dimension")
        ax.set_ylabel("VAMP-2 score")
        ax.legend()

In [ ]:
if run_vamp_analysis:
    # More in-depth analysis for dihedrals only, varying lag time and dimension.


    trajs = [[dihedrals]]  # your raw feature trajectories
    traj_names = ["dihedrals"]
    results = {}

    for n_vamp in [2, 3, 4, 5, 6, 7, 8,  9, 10]:
        for lagtime in [100, 500, 1000]:
            print(f"Evaluating lagtime = {lagtime}...")
            for i,traj in enumerate(trajs):
                fit_fetch = make_fit_fetch(n_vamp, lagtime)
                scores = vamp_score_cv(fit_fetch, trajs=traj, blocksize=lagtime, n=5, r=2)
                results[(n_vamp, lagtime)] = (scores.mean(), scores.std())
                print(f" dim: {n_vamp}, lagtime={lagtime}: VAMP-2 = {scores.mean():.4f} ± {scores.std():.4f}")





    rows = []

    for (n_vamp, lag), score in results.items():
        rows.append({
            "n_vamp": n_vamp,
            "lag_frames": lag,
            "score": score[0],
            "std": score[1]
        })

    df = pd.DataFrame(rows)

    df.to_csv("intermediate_outputs/vamp_feature_selection_dihedrals_only.csv", index=False)
    df.head()
    fig, ax = plt.subplots(1, figsize=(5, 3), sharey=True)

    for lag in df["lag_frames"].unique():
        temp = df[df["lag_frames"] == lag]
        mean = temp["score"]

        ax.plot(temp["n_vamp"], temp["score"], marker = "o", label=f"lag={lag}")
        ax.fill_between(temp["n_vamp"], temp["score"] - temp["std"], temp["score"] + temp["std"], alpha=0.2)

    ax.set_title(f"features: dihedrals")
    ax.set_xlabel("VAMP dimension")
    ax.set_ylabel("VAMP-2 score")
    ax.legend()